# Broderick 2018 自然语音 EEG 数据集云端下载与合约校验

本笔记本用于在 Google Colab CPU 运行时把 **Broderick et al. (2018) *The Old Man and the Sea*** 自然语音 EEG 原始数据集下载并校验到 Google Drive：

- **目标根目录**：`/content/drive/MyDrive/BrainDecoding/data/Broderick2018_NaturalSpeech`
- **双源数据包契约**：
  1. **脑电信号源（Dryad DOI `10.5061/dryad.070jc`）**：仅下载 `Natural Speech.zip`（`file_id=222587`，精确大小 `3,461,482,755` 字节，MD5 `5b827db6986b8c8eb22e2c45534b4693`）。**严禁**混入 `Cocktail Party.zip`、`Speech in Noise.zip`、`Natural Speech - Reverse.zip` 或 `N400.zip`。
  2. **词级强制对齐与带标点原文源（`private.zip`）**：从 `1UAegPighc2t48CWpfBhjAdsZg1VlJ-mZ` 下载并解压 `private/align1.json..align20.json` 与 `private/oldman_run1.txt..oldman_run20.txt`。
- **最终产物**：380 个 `.mat`（`Subject1..Subject19` × `Run1..Run20`）+ 20 个 `align*.json` + 20 个 `oldman_run*.txt`，并通过 `fs == 128.0 Hz`、`eegData [T, 128]`、`mastoids [T, 2]` 采样校验。

> **Dryad 授权下载说明**：Dryad 自 2024 年起对 API `/download` 与匿名下载端点启用授权/浏览器预签名校验。请在 Cell 3 选择三种合规路径之一：
> - **方式 A（推荐）**：在浏览器打开 Dryad 页面点击下载 `Natural Speech.zip`，右键下载项复制真实 S3 预签名链接填入 `DRYAD_PRESIGNED_URL`；
> - **方式 B**：填入官方 Dryad API v2 OAuth2 `DRYAD_OAUTH_CLIENT_ID` 与 `DRYAD_OAUTH_CLIENT_SECRET`；
> - **方式 C**：把提前下载好的 `Natural Speech.zip` 上传至 `Drive/BrainDecoding/archives/`、`TARGET_DIR/_downloads/` 或通过 `LOCAL_OR_DRIVE_ZIP_PATH` 指定。
        

In [ ]:
import os
import shutil
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/BrainDecoding")  # @param {type:"string"}
TARGET_DIR = DRIVE_ROOT / "data/Broderick2018_NaturalSpeech"
DOWNLOAD_STAGE_DIR = Path("/content/broderick_download_stage")
DRIVE_ZIP_BACKUP_DIR = TARGET_DIR / "_downloads"

FORCE_REDOWNLOAD = False  # @param {type:"boolean"}
CLEANUP_ARCHIVES = False  # @param {type:"boolean"}

TARGET_DIR.mkdir(parents=True, exist_ok=True)
DOWNLOAD_STAGE_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_ZIP_BACKUP_DIR.mkdir(parents=True, exist_ok=True)

local_usage = shutil.disk_usage("/content")
drive_usage = shutil.disk_usage(str(DRIVE_ROOT))
print(f"目标目录: {TARGET_DIR}")
print(f"Colab 本地可用空间: {local_usage.free / (1024**3):.2f} GB")
print(f"Google Drive 可用空间: {drive_usage.free / (1024**3):.2f} GB (建议至少 12.0 GB)")
if drive_usage.free < 11 * (1024**3):
    print("[警告] Google Drive 剩余空间偏紧；建议保留至少 12 GB，并在完成后可开启 CLEANUP_ARCHIVES=True。")
        

In [ ]:
import hashlib
import json
import shutil
import subprocess
import sys
import urllib.request
import zipfile
from pathlib import Path

PRIVATE_ZIP_GDRIVE_ID = "1UAegPighc2t48CWpfBhjAdsZg1VlJ-mZ"
PRIVATE_ZIP_NAME = "private.zip"


def _private_files_complete(target_dir: Path) -> bool:
    priv = target_dir / "private"
    return all((priv / f"align{r}.json").is_file() for r in range(1, 21)) and all(
        (priv / f"oldman_run{r}.txt").is_file() for r in range(1, 21)
    )


def _download_private_zip(dest_zip: Path) -> None:
    if dest_zip.is_file() and zipfile.is_zipfile(dest_zip) and not FORCE_REDOWNLOAD:
        print(f"复用已存在的 private.zip: {dest_zip}")
        return
    try:
        import gdown  # noqa: F401
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gdown"])
    import gdown

    url = f"https://drive.google.com/uc?id={PRIVATE_ZIP_GDRIVE_ID}"
    tmp_zip = dest_zip.with_suffix(".zip.tmp")
    tmp_zip.unlink(missing_ok=True)
    ok = gdown.download(url, str(tmp_zip), quiet=False, fuzzy=True)
    if not ok or not tmp_zip.is_file() or not zipfile.is_zipfile(tmp_zip):
        direct_url = (
            f"https://drive.usercontent.google.com/download?id={PRIVATE_ZIP_GDRIVE_ID}"
            "&export=download&confirm=t"
        )
        req = urllib.request.Request(
            direct_url, headers={"User-Agent": "BrainDecoding-Colab/1.0"}
        )
        with urllib.request.urlopen(req, timeout=120) as resp, tmp_zip.open("wb") as out:
            shutil.copyfileobj(resp, out, length=1024 * 1024)
    if not zipfile.is_zipfile(tmp_zip):
        raise RuntimeError(
            f"下载的 {PRIVATE_ZIP_NAME} 不是有效的 ZIP 归档；请检查 Google Drive 文件权限或重试。"
        )
    tmp_zip.replace(dest_zip)


def _extract_private_zip(zip_path: Path, target_dir: Path) -> None:
    private_dir = target_dir / "private"
    private_dir.mkdir(parents=True, exist_ok=True)
    needed = {f"align{r}.json" for r in range(1, 21)} | {
        f"oldman_run{r}.txt" for r in range(1, 21)
    }
    extracted = 0
    with zipfile.ZipFile(zip_path, "r") as zf:
        for info in zf.infolist():
            if info.is_dir():
                continue
            rel_parts = [p for p in Path(info.filename).parts if p and p != "."]
            if not rel_parts or any(p.startswith("__MACOSX") for p in rel_parts):
                continue
            basename = rel_parts[-1]
            if basename not in needed:
                continue
            dst = private_dir / basename
            if dst.is_file() and dst.stat().st_size == info.file_size and not FORCE_REDOWNLOAD:
                continue
            tmp_dst = dst.with_suffix(dst.suffix + ".tmp")
            with zf.open(info, "r") as src, tmp_dst.open("wb") as out:
                shutil.copyfileobj(src, out, length=1024 * 1024)
            tmp_dst.replace(dst)
            extracted += 1
    print(f"private.zip 解压完成（新写入 {extracted} 个文件）。")


if _private_files_complete(TARGET_DIR) and not FORCE_REDOWNLOAD:
    print(f"private/ 目录 20 个 align*.json 与 20 个 oldman_run*.txt 已就绪：{TARGET_DIR / 'private'}")
else:
    private_zip_path = DRIVE_ZIP_BACKUP_DIR / PRIVATE_ZIP_NAME
    _download_private_zip(private_zip_path)
    _extract_private_zip(private_zip_path, TARGET_DIR)
    if not _private_files_complete(TARGET_DIR):
        raise RuntimeError(f"private.zip 解压后文件不完整：{TARGET_DIR / 'private'}")
    print("private/ 完整性校验通过（20 align JSON + 20 oldman_run TXT）。")
        

In [ ]:
import hashlib
import json
import os
import shutil
import subprocess
import urllib.parse
import urllib.request
import zipfile
from pathlib import Path

DRYAD_DOI = "10.5061/dryad.070jc"
DRYAD_FILE_ID = 222587
NATURAL_SPEECH_ZIP_NAME = "Natural Speech.zip"
NATURAL_SPEECH_ZIP_BYTES = 3461482755
NATURAL_SPEECH_ZIP_MD5 = "5b827db6986b8c8eb22e2c45534b4693"

DRYAD_PRESIGNED_URL = ""  # @param {type:"string"}
DRYAD_OAUTH_CLIENT_ID = ""  # @param {type:"string"}
DRYAD_OAUTH_CLIENT_SECRET = ""  # @param {type:"string"}
LOCAL_OR_DRIVE_ZIP_PATH = ""  # @param {type:"string"}


def _md5_file(path: Path, chunk_bytes: int = 8 * 1024 * 1024) -> str:
    digest = hashlib.md5()
    with path.open("rb") as fh:
        while True:
            chunk = fh.read(chunk_bytes)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def _verify_natural_speech_zip(path: Path) -> tuple[bool, str]:
    if not path.is_file():
        return False, "文件不存在"
    size = path.stat().st_size
    if size != NATURAL_SPEECH_ZIP_BYTES:
        return False, f"大小不匹配：当前 {size} != 期望 {NATURAL_SPEECH_ZIP_BYTES}"
    if not zipfile.is_zipfile(path):
        return False, "不是有效 ZIP 文件"
    md5 = _md5_file(path)
    if md5 != NATURAL_SPEECH_ZIP_MD5:
        return False, f"MD5 不匹配：{md5} != {NATURAL_SPEECH_ZIP_MD5}"
    return True, "OK"


def _fetch_dryad_bearer_token(client_id: str, client_secret: str) -> str:
    body = urllib.parse.urlencode(
        {
            "client_id": client_id.strip(),
            "client_secret": client_secret.strip(),
            "grant_type": "client_credentials",
        }
    ).encode("utf-8")
    req = urllib.request.Request(
        "https://datadryad.org/oauth/token",
        data=body,
        headers={"Content-Type": "application/x-www-form-urlencoded"},
        method="POST",
    )
    with urllib.request.urlopen(req, timeout=60) as resp:
        payload = json.loads(resp.read().decode("utf-8"))
    token = payload.get("access_token", "").strip()
    if not token:
        raise RuntimeError(f"Dryad OAuth token 响应缺少 access_token: {payload}")
    return token


def _stream_download(url: str, dest: Path, headers: dict[str, str] | None = None) -> None:
    cmd = ["curl", "-L", "-C", "-", "--fail", "--progress-bar", "-o", str(dest)]
    for k, v in (headers or {}).items():
        cmd.extend(["-H", f"{k}: {v}"])
    cmd.append(url)
    subprocess.check_call(cmd)


def _locate_existing_zip() -> Path | None:
    candidates = [
        Path(LOCAL_OR_DRIVE_ZIP_PATH).expanduser() if LOCAL_OR_DRIVE_ZIP_PATH.strip() else None,
        DRIVE_ZIP_BACKUP_DIR / NATURAL_SPEECH_ZIP_NAME,
        DRIVE_ROOT / "archives" / NATURAL_SPEECH_ZIP_NAME,
        TARGET_DIR / NATURAL_SPEECH_ZIP_NAME,
        DOWNLOAD_STAGE_DIR / NATURAL_SPEECH_ZIP_NAME,
        Path("/content") / NATURAL_SPEECH_ZIP_NAME,
    ]
    for cand in candidates:
        if cand is None or not cand.is_file():
            continue
        ok, msg = _verify_natural_speech_zip(cand)
        if ok:
            return cand
        print(f"跳过未通过校验的候选 ZIP {cand}: {msg}")
    return None


NATURAL_SPEECH_ZIP_PATH = None if FORCE_REDOWNLOAD else _locate_existing_zip()
if NATURAL_SPEECH_ZIP_PATH is not None:
    print(f"复用已校验的 Natural Speech.zip: {NATURAL_SPEECH_ZIP_PATH}")
else:
    staged_zip = DOWNLOAD_STAGE_DIR / NATURAL_SPEECH_ZIP_NAME
    if DRYAD_PRESIGNED_URL.strip():
        print("通过浏览器 S3 预签名 URL 下载 Natural Speech.zip（支持断点续传）...")
        _stream_download(DRYAD_PRESIGNED_URL.strip(), staged_zip)
    elif DRYAD_OAUTH_CLIENT_ID.strip() and DRYAD_OAUTH_CLIENT_SECRET.strip():
        print("通过 Dryad API v2 OAuth2 client_credentials 下载 Natural Speech.zip...")
        token = _fetch_dryad_bearer_token(
            DRYAD_OAUTH_CLIENT_ID, DRYAD_OAUTH_CLIENT_SECRET
        )
        _stream_download(
            f"https://datadryad.org/api/v2/files/{DRYAD_FILE_ID}/download",
            staged_zip,
            headers={"Authorization": f"Bearer {token}"},
        )
    else:
        raise RuntimeError(
            "未找到已校验的 Natural Speech.zip，且未提供授权下载凭据。\n"
            "请任选以下一种合规方式后重新运行本单元：\n"
            "1) 在浏览器打开 https://datadryad.org/dataset/doi:10.5061/dryad.070jc ，"
            "点击下载 Natural Speech.zip（3.46 GB），从浏览器下载列表复制 S3 预签名链接填入 DRYAD_PRESIGNED_URL；\n"
            "2) 填入 Dryad 官方 OAuth2 凭据（DRYAD_OAUTH_CLIENT_ID / DRYAD_OAUTH_CLIENT_SECRET）；\n"
            f"3) 将下载好的 Natural Speech.zip 上传到 {DRIVE_ZIP_BACKUP_DIR} "
            f"或 {DRIVE_ROOT / 'archives'}（或填入 LOCAL_OR_DRIVE_ZIP_PATH）。"
        )

    ok, msg = _verify_natural_speech_zip(staged_zip)
    if not ok:
        raise RuntimeError(f"下载后的 Natural Speech.zip 校验失败：{msg}")
    drive_zip = DRIVE_ZIP_BACKUP_DIR / NATURAL_SPEECH_ZIP_NAME
    tmp_drive_zip = drive_zip.with_suffix(".zip.part")
    shutil.copy2(staged_zip, tmp_drive_zip)
    tmp_drive_zip.replace(drive_zip)
    NATURAL_SPEECH_ZIP_PATH = drive_zip
    print(f"Natural Speech.zip 校验并落盘完成：{NATURAL_SPEECH_ZIP_PATH}")
        

In [ ]:
import json
import re
import shutil
import zipfile
from pathlib import Path

from scipy.io import loadmat

MAT_PATTERN = re.compile(
    r"(?:^|/)Subject(?P<subj>1[0-9]|[1-9])/Subject(?P=subj)_Run(?P<run>20|1[0-9]|[1-9])\.mat$"
)
ALLOWED_SUBJECTS = set(range(1, 20))
ALLOWED_RUNS = set(range(1, 21))


def _canonical_mat_dest(member_name: str, target_dir: Path) -> Path | None:
    norm = member_name.replace("\\", "/")
    if norm.startswith("__MACOSX/") or any(
        blocked in norm
        for blocked in ("Cocktail Party", "Speech in Noise", "Reverse", "N400")
    ):
        return None
    match = MAT_PATTERN.search(norm)
    if not match:
        return None
    subj = int(match.group("subj"))
    run = int(match.group("run"))
    if subj not in ALLOWED_SUBJECTS or run not in ALLOWED_RUNS:
        return None
    return (
        target_dir
        / "Natural Speech"
        / "EEG"
        / f"Subject{subj}"
        / f"Subject{subj}_Run{run}.mat"
    )


extracted_count = 0
reused_count = 0
with zipfile.ZipFile(NATURAL_SPEECH_ZIP_PATH, "r") as zf:
    for info in zf.infolist():
        if info.is_dir():
            continue
        dst = _canonical_mat_dest(info.filename, TARGET_DIR)
        if dst is None:
            continue
        dst.parent.mkdir(parents=True, exist_ok=True)
        if dst.is_file() and dst.stat().st_size == info.file_size and not FORCE_REDOWNLOAD:
            reused_count += 1
            continue
        tmp_dst = dst.with_suffix(".mat.tmp")
        with zf.open(info, "r") as src, tmp_dst.open("wb") as out:
            shutil.copyfileobj(src, out, length=2 * 1024 * 1024)
        tmp_dst.replace(dst)
        extracted_count += 1
        if extracted_count % 40 == 0:
            print(f"已提取 {extracted_count} 个新 .mat 文件...")

print(f"解压统计：新写入 {extracted_count} 个，断点复用 {reused_count} 个。")

# 全量合约审计
missing_mats = []
for s in range(1, 20):
    for r in range(1, 21):
        mat_path = TARGET_DIR / "Natural Speech/EEG" / f"Subject{s}" / f"Subject{s}_Run{r}.mat"
        if not mat_path.is_file() or mat_path.stat().st_size == 0:
            missing_mats.append(str(mat_path))
if missing_mats:
    raise RuntimeError(f"缺失 {len(missing_mats)} 个 .mat 文件，示例: {missing_mats[:5]}")

for s, r in [(1, 1), (19, 20)]:
    sample_path = TARGET_DIR / "Natural Speech/EEG" / f"Subject{s}" / f"Subject{s}_Run{r}.mat"
    mat = loadmat(sample_path, variable_names=["fs", "eegData", "mastoids"])
    fs_val = float(mat["fs"].squeeze())
    eeg_shape = tuple(mat["eegData"].shape)
    mastoid_shape = tuple(mat["mastoids"].shape)
    assert abs(fs_val - 128.0) < 1e-6, f"期望 fs=128.0 Hz，实际 {fs_val}"
    assert len(eeg_shape) == 2 and eeg_shape[1] == 128, f"异常 eegData shape: {eeg_shape}"
    assert len(mastoid_shape) == 2 and mastoid_shape[1] == 2, f"异常 mastoids shape: {mastoid_shape}"
    print(f"抽检通过 {sample_path.name}: fs={fs_val} Hz, eegData={eeg_shape}, mastoids={mastoid_shape}")

for r in range(1, 21):
    align_path = TARGET_DIR / "private" / f"align{r}.json"
    text_path = TARGET_DIR / "private" / f"oldman_run{r}.txt"
    payload = json.loads(align_path.read_text(encoding="utf-8"))
    assert isinstance(payload.get("words"), list) and len(payload["words"]) > 0, f"{align_path} 缺少 words"
    assert text_path.read_text(encoding="utf-8").strip(), f"{text_path} 为空"

if CLEANUP_ARCHIVES:
    for z in (DRIVE_ZIP_BACKUP_DIR / "Natural Speech.zip", DRIVE_ZIP_BACKUP_DIR / "private.zip"):
        if z.is_file():
            z.unlink()
            print(f"已清理归档副本: {z}")
    shutil.rmtree(DOWNLOAD_STAGE_DIR, ignore_errors=True)

print("\n[完成] Broderick 2018 双源数据集全量就绪：380 个 .mat + 20 align JSON + 20 oldman_run TXT。")
        